# Imports

In [1]:
import numpy as np
import pandas as pd
import os
import sys
import time

In [2]:
import sklearn.tree
import sklearn.linear_model
import sklearn.metrics
import sklearn.ensemble

In [3]:
from pretty_print_sklearn_tree import pretty_print_sklearn_tree

In [4]:
# Plotting utils
import matplotlib
import matplotlib.pyplot as plt

plt.style.use('seaborn-v0_8') # pretty matplotlib plots

import seaborn as sns
sns.set('notebook', font_scale=1.25, style='whitegrid')

# Load all data from train/valid/test

In [5]:
data_dir = 'data_product_reviews/'
# data_dir = './' # for colab

### Load training

In [6]:
x_tr_NF = pd.read_csv(os.path.join(data_dir, 'x_train.csv')).values
y_tr_N = pd.read_csv(os.path.join(data_dir, 'y_train.csv')).values.flatten()

print("Training Data Shapes:")
print(x_tr_NF.shape)
print(y_tr_N.shape)

Training Data Shapes:
(6346, 7729)
(6346,)


### Load validation set

In [7]:
x_va_NF = pd.read_csv(os.path.join(data_dir, 'x_valid.csv')).values
y_va_N = pd.read_csv(os.path.join(data_dir, 'y_valid.csv')).values.flatten()

print("Validation Data Shapes:")
print(x_va_NF.shape)
print(y_va_N.shape)

Validation Data Shapes:
(792, 7729)
(792,)


### Load test set

In [8]:
x_te_NF = pd.read_csv(os.path.join(data_dir, 'x_test.csv')).values
y_te_N = pd.read_csv(os.path.join(data_dir, 'y_test.csv')).values.flatten()

print("Test Data Shapes:")
print(x_te_NF.shape)
print(y_te_N.shape)

Test Data Shapes:
(793, 7729)
(793,)


### Load vocabulary as a list of strings

In [9]:
x_df_for_vocab = pd.read_csv(os.path.join(data_dir, 'x_test.csv'))
vocab_list = x_df_for_vocab.columns.tolist()

print(f"Vocabulary extracted from CSV headers. Size: {len(vocab_list)}")
print("Example feature names from headers:", vocab_list[:5])

Vocabulary extracted from CSV headers. Size: 7729
Example feature names from headers: ['good', 'great', 'time', 'book', "don't"]


### Pack training and validation sets into big arrays (so we can use sklearn's hyperparameter search tools)

In [10]:
from sklearn.model_selection import PredefinedSplit

In [11]:
# vertically stack features
x_tr_va_NF = np.vstack([x_tr_NF, x_va_NF])

# horizontally stack labels
y_tr_va_N = np.hstack([y_tr_N, y_va_N])

In [12]:
# -1 (training), 0 (validation)
split_indicator = np.hstack([-1 * np.ones(x_tr_NF.shape[0]), np.zeros(x_va_NF.shape[0])])

my_splitter = PredefinedSplit(test_fold=split_indicator)

print("Combined Data Shapes:")
print(x_tr_va_NF.shape)
print(y_tr_va_N.shape)

Combined Data Shapes:
(7138, 7729)
(7138,)


# Problem 1: Decision Trees

## 1A: Train a simple tree with depth 3

In [13]:
simple_tree = sklearn.tree.DecisionTreeClassifier(
    criterion='gini',
    max_depth=3,
    min_samples_leaf=1,
    min_samples_split=2,
    random_state=101)

### **Fit the tree**

In [14]:
simple_tree = simple_tree.fit(x_tr_NF, y_tr_N)

### **Figure 1: Print Tree**

Use a helper function from the starter code

In [15]:
pretty_print_sklearn_tree(simple_tree, feature_names=vocab_list)

The binary tree structure has 15 nodes.
- depth   0 has    1 nodes, of which    0 are leaves
- depth   1 has    2 nodes, of which    0 are leaves
- depth   2 has    4 nodes, of which    0 are leaves
- depth   3 has    8 nodes, of which    8 are leaves
The decision tree:  (Note: Y = 'yes' to above question; N = 'no')
Decision: X['great'] <= 0.50?
  Y Decision: X['excel'] <= 0.50?
    Y Decision: X['disappoint'] <= 0.50?
      Y Leaf: p(y=1 | this leaf) = 0.430 (4041 total training examples)
      N Leaf: p(y=1 | this leaf) = 0.114 (368 total training examples)
    N Decision: X['disappoint'] <= 0.50?
      Y Leaf: p(y=1 | this leaf) = 0.903 (277 total training examples)
      N Leaf: p(y=1 | this leaf) = 0.429 (14 total training examples)
  N Decision: X['return'] <= 0.50?
    Y Decision: X['bad'] <= 0.50?
      Y Leaf: p(y=1 | this leaf) = 0.745 (1413 total training examples)
      N Leaf: p(y=1 | this leaf) = 0.415 (142 total training examples)
    N Decision: X['movie'] <= 0.50?
    

There is an internal node for excel > 0.5 with decision 'disappoint <= 0.5?', that has two child leaf nodes corresponding to the same sentiment class, i.e. both predict 0 (rounding down).

This doesn't make sense intuitively, but it does still accomplish the main goal of the tree to make the resulting groups as **pure** as possible. The algorithm doesn't care about the final classification it provides, only about reducing the Gini impurity.

It is likely that before the split, the parent node for this split contains a mix of positive and negative reviews. But after the split, even though both children predict negative, one leaf is more purely negative than the other (lower p(y=1) = 0.114). Thus, the split isolates a group of reviews that are very likely to be negative from a group that is only slightly more mixed.

## 1B : Find best Decision Tree with grid search

In [16]:
from sklearn.model_selection import GridSearchCV

# base tree for search
base_tree = sklearn.tree.DecisionTreeClassifier(criterion='gini', random_state=101)

param_grid = {
    'max_depth': [2, 8, 32, 128],
    'min_samples_leaf': [1, 3, 9],
} # only one random_state given eh

searcher = GridSearchCV(
    base_tree,
    param_grid,
    scoring='balanced_accuracy',
    cv=my_splitter, # use our fixed train/validation split
    return_train_score=True,
    refit=False) # won't retrain a final model on all data

print("Running grid search...")
# Fit the searcher on the combined training and validation data
searcher.fit(x_tr_va_NF, y_tr_va_N)
print("Done.")

Running grid search...
Done.


In [17]:
results_df = pd.DataFrame(searcher.cv_results_)
print("Grid Search Results:")
display(results_df[['param_max_depth', 'param_min_samples_leaf', 'mean_test_score', 'mean_train_score']])

best_row_id = results_df['mean_test_score'].idxmax()
best_params_ = results_df.loc[best_row_id, 'params']

print("\nBest Hyperparameters:")
print(best_params_)

Grid Search Results:


,param_max_depth,param_min_samples_leaf,mean_test_score,mean_train_score
0,2,1,0.633931,0.640429
1,2,3,0.633931,0.640429
2,2,9,0.633931,0.640429
3,8,1,0.698887,0.726505
4,8,3,0.697599,0.721935
5,8,9,0.696208,0.712791
6,32,1,0.727378,0.918197
7,32,3,0.732380,0.877218
8,32,9,0.713458,0.814975
9,128,1,0.730313,0.996216



Best Hyperparameters:
{'max_depth': 32, 'min_samples_leaf': 3}


### Build the best decision tree

**TODO Build the Best Tree on the training set** in the next coding cell



In [18]:
best_tree = base_tree # TODO call set_params using the best_params_ found by your searcher
best_tree.set_params(**best_params_)
best_tree.fit(x_tr_NF, y_tr_N)

DecisionTreeClassifier(max_depth=32, min_samples_leaf=3, random_state=101)

### Interpret the best decision tree

In [19]:
pretty_print_sklearn_tree(best_tree, feature_names=vocab_list)

The binary tree structure has 817 nodes.
- depth   0 has    1 nodes, of which    0 are leaves
- depth   1 has    2 nodes, of which    0 are leaves
- depth   2 has    4 nodes, of which    0 are leaves
- depth   3 has    8 nodes, of which    0 are leaves
- depth   4 has   16 nodes, of which    6 are leaves
- depth   5 has   20 nodes, of which    5 are leaves
- depth   6 has   30 nodes, of which   14 are leaves
- depth   7 has   32 nodes, of which   18 are leaves
- depth   8 has   28 nodes, of which   12 are leaves
- depth   9 has   32 nodes, of which   11 are leaves
- depth  10 has   42 nodes, of which   21 are leaves
- depth  11 has   42 nodes, of which   23 are leaves
- depth  12 has   38 nodes, of which   17 are leaves
- depth  13 has   42 nodes, of which   21 are leaves
- depth  14 has   42 nodes, of which   25 are leaves
- depth  15 has   34 nodes, of which   18 are leaves
- depth  16 has   32 nodes, of which   14 are leaves
- depth  17 has   36 nodes, of which   23 are leaves
- dep

# Problem 2: Random forest

## 2A: Train a random forest with default settings

In [20]:
simple_forest = sklearn.ensemble.RandomForestClassifier(
    n_estimators=100,
    criterion='gini',
    max_features='sqrt',
    max_depth=3,
    min_samples_leaf=1,
    min_samples_split=2,
    random_state=101)

### Fit the forest

**TODO Train on the training set** in the next coding cell

In [21]:
simple_forest = simple_forest.fit(x_tr_NF, y_tr_N)

## 2B & Table 2: Feature Importances

### Table 2
**Sample Output** (Feel free to print all words and organize them in any software)

|**Important Words**|**Unimportant Words**|
|:-:|:-:|
|I1 |  U1  |
|I2 |  U2  |
|I3 |  U3  |
|I4 |  U4  |
|I5 |  U5  |
|I6 |  U6  |
|I7 |  U7  |
|I8 |  U8  |
|I9 |  U9  |
|I0 |  U0  |

In [22]:
importances = simple_forest.feature_importances_

feature_importance_df = pd.DataFrame({
    'feature': vocab_list,
    'importance': importances
})

# top 10 most important
top_10_features = feature_importance_df.sort_values(by='importance', ascending=False).head(10)

# 10 randomly from near-zero importance
unimportant_features_df = feature_importance_df[feature_importance_df['importance'] < 0.00001]
num_unimportant = len(unimportant_features_df)

random_10_unimportant = unimportant_features_df.sample(10, random_state=101) # reproducible

comparison_table = pd.DataFrame({
    'Important Words': top_10_features['feature'].values,
    'Unimportant Words': random_10_unimportant['feature'].values
})

# display comparison
print("Feature Importance Analysis (Simple Random Forest):")
display(comparison_table)
print(f"\nNumber of features with near-zero importance (< 0.00001): {num_unimportant}")

Feature Importance Analysis (Simple Random Forest):


,Important Words,Unimportant Words
0,return,doing_the
1,excel,bought_these
2,great,mix
3,worst,not_all
4,poor,vision
5,disappoint,example_the
6,your_money,the_author's
7,i_love,abuse
8,the_best,went_on
9,don't,also_very



Number of features with near-zero importance (< 0.00001): 7296


## 2C: Best Random Forest via grid search



This block might take 2-10 minutes.

If yours runs significantly longer, try this out on Google Colab instead.

In [23]:
# base random forest for search
base_forest = sklearn.ensemble.RandomForestClassifier(
    n_estimators=100,
    min_samples_leaf=1,
    random_state=101)

forest_param_grid = {
    'max_features': [3, 10, 33, 100, 333],
    'max_depth': [16, 32],
}
# rest are fixed only eh

forest_searcher = GridSearchCV(
    base_forest,
    forest_param_grid,
    scoring='balanced_accuracy',
    cv=my_splitter,
    return_train_score=True,
    refit=False,
    verbose=1) # progress updates

print("Running grid search for Random Forest...")
forest_searcher.fit(x_tr_va_NF, y_tr_va_N)
print("Done.")

Running grid search for Random Forest...
Fitting 1 folds for each of 10 candidates, totalling 10 fits
Done.


In [24]:
forest_results_df = pd.DataFrame(forest_searcher.cv_results_)
print("\nGrid Search Results (Random Forest):")
display(forest_results_df[['param_max_features', 'param_max_depth', 'mean_test_score', 'mean_train_score']])

best_forest_row_id = forest_results_df['mean_test_score'].idxmax()
best_forest_params_ = forest_results_df.loc[best_forest_row_id, 'params']

print("\nBest Hyperparameters for Random Forest:")
print(best_forest_params_)


Grid Search Results (Random Forest):


,param_max_features,param_max_depth,mean_test_score,mean_train_score
0,3,16,0.751761,0.908283
1,10,16,0.840997,0.925010
2,33,16,0.842439,0.923600
3,100,16,0.839402,0.920745
4,333,16,0.784194,0.871229
5,3,32,0.796507,0.968174
6,10,32,0.843626,0.970070
7,33,32,0.851204,0.964402
8,100,32,0.835434,0.969748
9,333,32,0.812187,0.952712



Best Hyperparameters for Random Forest:
{'max_depth': 32, 'max_features': 33}


The value of max_features is 33 for my best forest.

The maximum possible value for this is the total number of features in the dataset, which in this case is 7729. Thus it is beneficial to tune this hyperparameter to control the bias-variance tradeoff in the random forest.

If max_features is too large, every tree in the forest will be very similar because they will all tend to pick the same few best features for their splits - the trees will be highly correlated, leading to overfitting and losing the whole point of the forest!

On the other hand, if max_features is too small, the individual trees will be too simple and weak due to the lack of good features to choose from in the splits (7296 features were useless anyway), leading to underfitting!

Thus we need to tune it to find a good middle point where the trees are different enough to be robuts, but also strong enough to be good individual predictors.

The primary tradeoff controlled by n_estimators (number of trees) is that between performance and computational cost. More trees would generally lead to a better and more stable model, but would require more memory and time to train! Similar to earlier kinds of models, after a point, adding more trees gives diminishing returns on performance while continuing to increase the training time.

One cannot generally overfit by setting n_estimators to be too large - while the training error might continue to go down, the error on a validation set will not start to increase again (characteristic of overfitting). This is an advantage of the algorithm, and happens because the model's overall prediction is an average of many de-correlated trees. Adding more trees makes the average stable without causing overfitting!

### Build the best random forest using the best hyperparameters found in 2B

This is necessary so you have the specific best performing forest in your workspace.

Train *only* on training set (do not merge train and valid)


In [25]:
best_forest = base_forest
best_forest.set_params(**best_forest_params_)
best_forest.fit(x_tr_NF, y_tr_N)

RandomForestClassifier(max_depth=32, max_features=33, random_state=101)

### Table 3: Comparison of methods on the bag-of-words to sentiment classification task.

Please report **balanced accuracy** on the train, valid, and test sets, to 3 digits of precision

**Sample Output** (Feel free to print all values and organize them by hand)

|**method**|**max depth**|**num trees**|**train BAcc**|**valid BAcc**|**test BAcc**|
|:-|:-:|:-:|:-:|:-:|:-:|
|simple Tree	| 1 | 1 | 0.123	|0.456	|0.890|
|best Tree	|1 | 1 | 0.123	|0.456	|0.890|
|simple RandomForest	|1 | 1 | 0.123	|0.456	|0.890|
|best RandomForest	|1 | 1 | 0.123	|0.456	|0.890|

In [27]:
models = {
    "Simple Tree": simple_tree,
    "Best Tree": best_tree,
    "Simple RandomForest": simple_forest,
    "Best RandomForest": best_forest
}

table_rows = []

for name, model in models.items():
    # hyperparams
    max_depth = model.get_params()['max_depth']
    num_trees = getattr(model, 'n_estimators', 1) # 1 for single trees

    # balanced accuracy on all 3 sets
    train_bacc = sklearn.metrics.balanced_accuracy_score(y_tr_N, model.predict(x_tr_NF))
    valid_bacc = sklearn.metrics.balanced_accuracy_score(y_va_N, model.predict(x_va_NF))
    test_bacc = sklearn.metrics.balanced_accuracy_score(y_te_N, model.predict(x_te_NF))

    table_rows.append({
        'method': name,
        'max depth': max_depth,
        'num trees': num_trees,
        'train BAcc': f"{train_bacc:.3f}",
        'valid BAcc': f"{valid_bacc:.3f}",
        'test BAcc': f"{test_bacc:.3f}"
    })

final_results_df = pd.DataFrame(table_rows)

display(final_results_df)

,method,max depth,num trees,train BAcc,valid BAcc,test BAcc
0,Simple Tree,3,1,0.646,0.645,0.646
1,Best Tree,32,1,0.877,0.732,0.749
2,Simple RandomForest,3,100,0.819,0.797,0.778
3,Best RandomForest,32,100,0.964,0.851,0.837


### Summary

The performance of the best tree is signiicantly better than that of the simple tree on the validation and test sets. The same goes for the best random forest. Thus, tuning the hyperparameters helps make a high-performing model.

Further, ensembles are powerful, as is evident from the performance of the best random forest (accuracy on test set is much higher than others)! This demonstrates the core benefit - combining many models that may not be very strong individually, to get a much more robust and accurate final model.

Additionally, the deeper models like best tree and best random forest, get very high accuracies on the training set (96 for random forest!). But the test accuracy of the best random forest is also very high and closer to its own training accuracy, rather than the test accuracy of a simple decision tree. This shows that the ensemble models are less prone to overfitting and generalize better.